# Trabajo en clase — Q-Learning con Taxi (Actividades)

Este notebook contiene exclusivamente la actividad y preguntas teóricas para Taxi-v4, junto con sus dependencias base.

## 1. Dependencias y base (`comun.py`)

In [ ]:
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

from IPython.display import HTML
from matplotlib import animation

env = gym.make("Taxi-v4", render_mode="rgb_array")

print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

state, info = env.reset(seed=42)
action = env.action_space.sample()
next_state, reward, terminated, truncated, info = env.step(action)

print("Estado:", state)
print("Acción:", action)
print("Nuevo estado:", next_state)
print("Recompensa:", reward)
print("Terminated:", terminated)

n_states = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))

print("Shape de Q:", Q.shape)
print(Q[:5])


## 2. Preguntas Teóricas

###  Pregunta 1
**¿Cuántos estados y cuántas acciones tiene Taxi-v4?**
- Tiene 500 estados y 6 acciones.

**Explique brevemente por qué Taxi tiene muchos más estados que FrozenLake.**
- En FrozenLake (4x4) el estado solo depende de la posición en la cuadrícula (16).
- En Taxi, el estado combina tres elementos: 
  1. La posición del taxi en la cuadrícula (5x5 = 25 posibles).
  2. La ubicación actual del pasajero (4 paradas posibles + 1 estado si está dentro del taxi = 5).
  3. El destino del pasajero (4 paradas posibles).
  En total: $25 \times 5 \times 4 = 500$ combinaciones (estados) posibles.


###  Pregunta 2
**¿Qué representa cada elemento devuelto por `env.step(action)`?**
- `state`: El estado actual del entorno antes de realizar la acción.
- `action`: La acción elegida para ejecutar (ej. moverse en alguna dirección, recoger o dejar).
- `next_state`: El estado resultante en el entorno después de realizar la acción.
- `reward`: La recompensa obtenida al realizar esa acción desde el estado previo.
- `terminated`: Un valor booleano (True/False) que indica si el episodio terminó (ej. el pasajero fue entregado en su destino).


###  Pregunta 3
**¿Cuántos valores debe aprender el agente en total?**
- Debe aprender un valor Q por cada par estado-acción. 
  Por lo tanto: $500 \text{ estados} \times 6 \text{ acciones} = 3,000$ valores totales.


## 3. Actividad 1 — Política Epsilon-Greedy

In [ ]:
def choose_action(Q, state, epsilon, env):
    """
    Política epsilon-greedy para seleccionar acciones
    """
    # 1. decidir si explorar o explotar
    if random.random() < epsilon:
        # Explorar: retornar una acción aleatoria
        return env.action_space.sample()
    else:
        # Explotar: retornar la mejor acción para el estado actual según Q
        q_values = Q[state]
        max_q = np.max(q_values)
        
        # En caso de empate entre varias acciones con el mismo valor máximo Q,
        # seleccionamos aleatoriamente entre ellas.
        best_actions = np.flatnonzero(q_values == max_q)
        return int(np.random.choice(best_actions))

# Prueba rápida de la función de Actividad 1
state, info = env.reset(seed=42)
epsilon = 0.1
action = choose_action(Q, state, epsilon, env)
print(f"Probando choose_action en el estado {state} con epsilon {epsilon} -> Acción seleccionada: {action}")


## 4. Actividad 2 — Actualizar Q 

In [ ]:

def update_q(Q, state, action, terminated, reward, next_state, alpha, gamma):
    
    current_q= Q[state][action]
     #si ya terminamos el episodio ya no hay valores de q futuros
    if terminated:
        best_next_q = 0.0
    else:
        best_next_q = np.max(Q[next_state])
        # Q es una matriz de numpy por lo tanto no se utiliza el .values() 
        #basta con solo poner el indice para acceder a la fila de acciones

    #TD TARGET
    td_target = reward + gamma * best_next_q
    #TD ERROR
    td_error = td_target - current_q

    Q[state, action]=( #manera de acceder a el valor que estamos actualizando
        current_q + alpha * td_error
    )
    

    